# GAN Implementation - My Learning Notes

trying to understand GANs properly... let me implement from scratch

date: march 2026

## what i understand so far:

GANs = two networks fighting each other
- generator makes fake stuff
- discriminator tries to catch the fake stuff

like a forger vs police detective

both get better over time

In [ ]:
# imports - need these for sure
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# check if gpu available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"using: {device}")

In [ ]:
# hyperparameters - copied from some tutorial but will experiment
lr = 0.0002  # learning rate
batch_size = 64
image_size = 64
channels = 1  # grayscale
noise_dim = 100  # random noise vector size
epochs = 20  # start small, see how it goes

print("params set")

In [ ]:
# get data - using MNIST cuz its simple
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])  # normalize to [-1, 1]
])

dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print(f"loaded {len(dataset)} images")

# lets see some real images first
real_batch = next(iter(dataloader))
print(f"batch shape: {real_batch[0].shape}")

## Generator Network

this takes noise and makes fake images

need to go from small noise vector to full image... how?
- use transpose convolutions (upsampling)
- start small, make bigger

lets try...

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        
        # hmm... need to go from 100 -> 64x64 image
        # 100 -> 4x4 -> 8x8 -> 16x16 -> 32x32 -> 64x64
        
        self.main = nn.Sequential(
            # input is noise_dim x 1 x 1
            nn.ConvTranspose2d(100, 512, 4, 1, 0, bias=False),  # -> 4x4
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),  # -> 8x8
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),  # -> 16x16
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),   # -> 32x32
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(64, 1, 4, 2, 1, bias=False),     # -> 64x64
            nn.Tanh()  # output [-1, 1] to match normalized data
        )
    
    def forward(self, x):
        return self.main(x)

# test it
gen = Generator().to(device)
test_noise = torch.randn(1, 100, 1, 1).to(device)
test_out = gen(test_noise)
print(f"generator output shape: {test_out.shape}")  # should be [1, 1, 64, 64]

## Discriminator

this one is easier - just a classifier
takes image -> outputs probability (real or fake)

basically reverse of generator

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        
        # 64x64 -> 32x32 -> 16x16 -> 8x8 -> 4x4 -> 1x1
        self.main = nn.Sequential(
            # input is 1 x 64 x 64
            nn.Conv2d(1, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # note: no batchnorm on first layer (read this somewhere)
            
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(512, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()  # probability output
        )
    
    def forward(self, x):
        return self.main(x).view(-1, 1).squeeze(1)

# test discriminator
disc = Discriminator().to(device)
test_img = torch.randn(1, 1, 64, 64).to(device)
test_out = disc(test_img)
print(f"discriminator output: {test_out.item():.4f}")  # should be between 0 and 1